In [1]:
"""Блок A: сырые CSV -> train_full.parquet (склад x артикул x день, с нулями и остатками).
Запуск: python prep.py sales_dataset_hakaton.csv stock_dataset_hakaton.csv"""
import sys
import numpy as np
import pandas as pd

KEY = ["date", "warehouse_name", "article"]
PAIR = ["warehouse_name", "article"]

In [2]:
def build(sales_path, stock_path, out="train_full.parquet"):
    s = pd.read_csv(sales_path, parse_dates=["date"])
    st = pd.read_csv(stock_path, parse_dates=["date"]).rename(columns={"char_article": "article"})

    # A1: что мы получили (сверить с ТЗ: 11 складов, 799 артикулов)
    print("sales:", s.shape, s.date.min().date(), "->", s.date.max().date())
    print("warehouses:", s.warehouse_name.nunique(), "articles:", s.article.nunique())
    print("NaN по колонкам:\n", s.isna().sum()[lambda x: x > 0])
    print("полных дублей строк:", s.duplicated().sum(), "| quantity<0:", (s.quantity < 0).sum())
    print("артикулов из stock нет в sales:", len(set(st.article) - set(s.article)))

    # A2: суммируем департаменты
    agg = {"quantity": "sum", "sales_sum": "sum", "discount_sum": "sum",
       "cost_price": "mean", "volume_weight": "mean"}
    daily = s.groupby(KEY, as_index=False).agg({c: f for c, f in agg.items() if c in s.columns})

    # A3: полная сетка 11 складов × 799 артикулов (столько и ждёт submission)
    pairs = pd.MultiIndex.from_product([s.warehouse_name.unique(), s.article.unique()], names=PAIR).to_frame(index=False)
    last = s.sort_values("date")
    cat_a = last.groupby("article")[["category_lvl_1", "category_lvl_2", "category_lvl_3"]].last().reset_index()
    city_w = last.groupby("warehouse_name").cities_name.last().reset_index()
    days = pd.DataFrame({"date": pd.date_range(s.date.min(), s.date.max())})
    grid = (pairs.merge(days, how="cross")
            .merge(daily, on=KEY, how="left")
            .merge(cat_a, on="article", how="left")
            .merge(city_w, on="warehouse_name", how="left"))
    fill = ["quantity", "sales_sum", "discount_sum"]
    grid[fill] = grid[fill].fillna(0)
    assert np.isclose(grid.quantity.sum(), s.quantity.sum()), "агрегация потеряла продажи"
    print("пар:", len(pairs), "строк в сетке:", len(grid))

    # A4: остатки. NaN НЕ заполняем молча - даём флаги, решение принимает модель
    st = st.groupby(KEY, as_index=False).stock.sum()
    grid = grid.merge(st, on=KEY, how="left")
    grid["stock_missing"] = grid.stock.isna()
    grid["stock_zero"] = grid.stock.eq(0)
    print("доля дней без записи в stock:", round(grid.stock_missing.mean(), 3),
          "| доля stock==0:", round(grid.stock_zero.mean(), 3))

    grid.to_parquet(out, index=False)
    return grid

In [3]:
grid = build("sales_dataset_hakaton_train.csv", "stock_dataset_hakaton_train.csv")

sales: (1492370, 12) 2024-01-01 -> 2026-07-31
warehouses: 11 articles: 799
NaN по колонкам:
 category_lvl_3    1491432
sales_sum               3
discount_sum       316955
dtype: int64
полных дублей строк: 0 | quantity<0: 0
артикулов из stock нет в sales: 0
пар: 8789 строк в сетке: 8288027
доля дней без записи в stock: 0.414 | доля stock==0: 0.345


In [4]:
g = grid
g["state"] = np.where(g.stock_missing, "missing", np.where(g.stock_zero, "zero", "positive"))
g["sold"] = g.quantity > 0
print(g.groupby("state").agg(rows=("quantity", "size"), mean_q=("quantity", "mean"), sold_share=("sold", "mean")))

p = g.groupby(PAIR).stock_missing.mean()
print("пар без единой записи stock:", (p == 1).sum(), "| без пропусков:", (p == 0).sum())
print(p.describe())
print(g.groupby(g.date.dt.to_period("Q")).stock_missing.mean())   # где во времени пропуски

             rows    mean_q  sold_share
state                                  
missing   3430196  0.033477    0.026503
positive  1995805  0.604748    0.277096
zero      2862026  0.176448    0.114163
пар без единой записи stock: 331 | без пропусков: 1530
count    8789.000000
mean        0.413874
std         0.326237
min         0.000000
25%         0.106045
50%         0.408271
75%         0.676564
max         1.000000
Name: stock_missing, dtype: float64
date
2024Q1    0.791690
2024Q2    0.719267
2024Q3    0.622736
2024Q4    0.538805
2025Q1    0.483181
2025Q2    0.373476
2025Q3    0.274381
2025Q4    0.197243
2026Q1    0.154269
2026Q2    0.107433
2026Q3    0.045207
Freq: Q-DEC, Name: stock_missing, dtype: float64


In [5]:
allp = pd.MultiIndex.from_product([g.warehouse_name.unique(), g.article.unique()])
have = pd.MultiIndex.from_frame(g[PAIR].drop_duplicates())
miss = allp.difference(have)
print(len(miss)); print(list(miss)[:12])

0
[]


In [6]:
no_stock = p[p == 1].index
print(g[g.set_index(PAIR).index.isin(no_stock)].quantity.mean(), g.quantity.mean())

0.014493821544021299 0.22041325396261377


In [7]:
print(grid.category_lvl_1.isna().sum(), grid.cities_name.isna().sum())

0 0
